### v2
- 分离reasoning, answer。添加 `<think>`, `</think>`标签
- 统计长度，取max_len = 95% 分位数
- 处理loss masking (不计算 question 部分交叉熵损失)
  - SFTTrainer直接读取example['messages']，用processing_class来apply_chat_template，并作loss masking
  - 也可以考虑Data Collator
- 启用tensorboard

### v3 适配 7B / 14B 模型, A10 24G
- 4-bit 量化
- Flash Attention 2
- 开启 Gradient Checkpointing
- 使用 Paged Optimizer （防止显存碎片导致的 OOM）
- DeepSpeed ZeRO-2 或 ZeRO-3 进行多卡并行
- 计算 & 验证 显卡能接受的最长CoT长度，预处理时过滤
- 数据集预处理后 save_to_disk
- accelerate / torch 启动命令

In [ ]:
import os
import pandas as pd
import numpy as np
import re

# Dataset
from datasets import Dataset, load_from_disk

# Base Model
from modelscope import snapshot_download, AutoModelForCausalLM, AutoTokenizer

# Fine Tuning
import torch
from transformers import TrainingArguments, IntervalStrategy
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer


In [ ]:
# Dataset
DATASET_DIR = "../Datasets"
CACHE_DIR = os.path.join(DATASET_DIR, "Cache")
DATASET_PATH_COT = os.path.join(DATASET_DIR, "NuminaMath-CoT")
DATASET_PATH_TIR = os.path.join(DATASET_DIR, "NuminaMath-TIR")
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
MAX_LEN = 6400 # 95%分位数

# Base Model
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B" 
MODEL_DIR = "../Models"

# Fine Tuning
LORA_COT_DIR = "./LoRA-CoT"     # 保存模型检查点、日志和配置的目录
LORA_TIR_DIR = "./LoRA-TIR"
LORA_COT_FINAL = os.path.join(LORA_COT_DIR, "DeepSeek-R1-Distill-Qwen-7B-CoT")
LORA_TIR_FINAL = os.path.join(LORA_COT_DIR, "DeepSeek-R1-Distill-Qwen-7B-TIR")
os.makedirs(LORA_COT_DIR, exist_ok=True)
os.makedirs(LORA_TIR_DIR, exist_ok=True)


NUM_TRAIN_EPOCHS = 2
BATCH_SIZE = 1
ACCU_STEPS = 5
NUM_GPUS = 2

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"使用 CUDA: {torch.cuda.get_device_name(0)}")
    # 如果有多个 GPU，可以使用 device_map="auto" 让 accelerate 自动分配
    # device_map = "auto" 
    dtype = torch.bfloat16 # NVIDIA Ampere+ 支持良好
    use_flash_attn = True
elif torch.backends.mps.is_available():
    device = "mps"
    print("使用 MPS")
    dtype = torch.float16 # MPS 上 float16 通常比 bfloat16 更稳定
    use_flash_attn = False # MPS 不支持 Flash Attn 2
else:
    device = "cpu"
    print("使用 CPU")
    dtype = torch.float32
    use_flash_attn = False

# Part 1 Base Model

In [ ]:
# 1.1 下载模型
model_path = snapshot_download(     # 如果目录已存在且完整，它会秒级返回，不会重复下载
    MODEL_ID, 
    cache_dir=MODEL_DIR, 
    revision='master'
)

In [ ]:
# 1.2 加载model和tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map=device, 
    trust_remote_code=True,
    dtype=torch.bfloat16, 
    # attn_implementation="flash_attention_2",    
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
print(f"{MODEL_ID} 加载成功！设备：{model.device}, 数据类型：{model.dtype}")
print(model)

# Part 2 Datasets

In [ ]:
# 2.1 下载数据集
if not os.path.exists(DATASET_PATH_COT):
    from datasets import load_dataset, DownloadConfig
    download_config = DownloadConfig(
        resume_download=True 
    )
    dataset_cot = load_dataset("AI-MO/NuminaMath-CoT", cache_dir=CACHE_DIR, download_config=download_config)

    dataset_cot.save_to_disk(DATASET_PATH_COT)

In [ ]:
# 2.2 加载数据集
# 长度分布百分位: {20: 1890, 50: 2739, 75:4027, 95: 6353, 99: 8051}
dataset_cot = load_from_disk(DATASET_PATH_COT)
indices = list(range(1000))
dataset_cot_train = dataset_cot["train"].select(indices)
dataset_cot_test = dataset_cot["test"]
print(dataset_cot)

In [ ]:
# 2.3 数据预处理
# 加上 <think>, </think> 标签
def format_deepseek_r1(example):
    """
    专门针对 DeepSeek-R1-Distill 格式的数据预处理：
    1. 提取 assistant 回复中包含 \boxed{} 的最后一部分作为最终答案。
    2. 将剩余部分包裹在 <thinking> 标签中。
    3. 重组为：<thinking>推理过程</thinking>\n\n最终答案
    """
    messages = example['messages']
    new_messages = []

    for msg in messages:
        if msg['role'] == 'assistant':
            content = msg['content'].strip()
            
            # 正则表达式匹配最后一个 \boxed{...} 及其所在的句子/段落
            # 逻辑：从后往前找，或者匹配包含 \boxed 的最后一段完整语义
            # 这里假设 \boxed 出现在最后，且前面可能有 "Thus", "Therefore" 等引导词
            
            # 策略：找到最后一个 \boxed{...} 的起始位置
            # 我们尝试匹配：任意字符 + (\n\n 或 \n) + (可选的引导词) + \boxed{...} + (结尾空白)
            # 为了稳健，我们直接寻找最后一个 \boxed 的位置，然后向前回溯找到合理的断句点
            
            last_boxed_match = None
            for match in re.finditer(r'\\boxed\{[^}]*\}', content):
                last_boxed_match = match
            
            if last_boxed_match:
                boxed_end_pos = last_boxed_match.end()
                
                # 向后截取：包含 \boxed 的部分以及它前面的引导句（直到遇到两个换行或句号）
                # 简单策略：从最后一个 \boxed 的开头往前找，直到遇到双换行符 \n\n 或者 句子结束符 (. ! ?)
                # 但更简单的做法是：假设 \boxed 所在的那一段（由 \n\n 分隔）就是答案段
                
                suffix_content = content[last_boxed_match.start():]
                
                # 向前寻找分割点：从 \boxed 开始往前找，直到遇到 \n\n (段落分隔)
                # 如果没有 \n\n，就尽量找一个句号，或者直接截断
                prefix_content = content[:last_boxed_match.start()]
                
                # 尝试在 prefix_content 末尾找最近的 \n\n 作为分割线
                # 如果找不到，说明思考和答案混在一个段落，这时候需要更细致的规则
                # 鉴于你的描述 "最后一句话为answer"，我们可以假设答案前通常有换行
                
                split_index = -1
                # 从后往前找第一个双换行
                last_double_newline = prefix_content.rfind('\n\n')
                # 如果没找到双换行，找单换行
                if last_double_newline == -1:
                    last_single_newline = prefix_content.rfind('\n')
                    if last_single_newline != -1:
                        split_index = last_single_newline
                else:
                    split_index = last_double_newline
                
                if split_index != -1:
                    thinking_part = prefix_content[:split_index].strip()
                    answer_part = prefix_content[split_index:].strip() + suffix_content
                else:
                    # 极端情况：没有明显换行，强行按 \boxed 前最近的一个句号分割，或者全部视为思考（不推荐）
                    # 这里做一个保守处理：假设 \boxed 前的一小段是答案，其余是思考
                    # 尝试找最后一个句号
                    last_period = prefix_content.rfind('.')
                    if last_period != -1:
                        thinking_part = prefix_content[:last_period+1].strip()
                        answer_part = prefix_content[last_period+1:].strip() + suffix_content
                    else:
                        # 实在分不开，将整个内容视为思考，并在末尾追加 \boxed (这可能不符合预期，需检查数据)
                        # 但根据你的描述，这种情况应该很少
                        thinking_part = content
                        answer_part = "" 
                        # 如果分不开，可能意味着数据本身格式就不太对，这里暂时保持原样并打标
                        # 实际上，如果分不开，最好把整个都放进 thinking，因为模型需要学习推导过程
                        # 但为了符合 R1 格式，我们尽量分离。
                        # 修正策略：如果无法分离，将整个内容放入 thinking，并在末尾显式加上 \boxed 部分（如果已经被包含）
                        # 这里的逻辑稍微复杂，针对你的明确描述，我们假设一定能找到换行
                        thinking_part = content
                        answer_part = ""

                # 如果成功分离了 answer_part
                if answer_part.strip():
                    final_content = f"<think>\n{thinking_part}\n</think>\n\n{answer_part.strip()}"
                else:
                    # 如果没能分离出答案部分（比如整段都是推导，\boxed 只是其中一部分），
                    # 按照 R1 的习惯，有时候整个回复都在 thinking 里，或者只有最后结果在外面。
                    # 既然你明确说 "最后一句话为 answer"，那上面逻辑应该能覆盖。
                    # 兜底方案：
                    final_content = f"<think>\n{content}\n</think>"
            else:
                # 如果没有 \boxed，说明这条数据可能不符合规范，整体视为思考
                final_content = f"<think>\n{content}\n</think>"
            
            new_messages.append({'role': 'assistant', 'content': final_content})
        else:
            new_messages.append(msg)
    
    example['messages'] = new_messages
    return example

In [ ]:
def is_within_length_limit(example):
    """
    检查对话总长度是否超过限制。
    返回 True 表示保留，False 表示过滤掉。
    """
    messages = example['messages']
    
    # 关键：必须使用 apply_chat_template 并 tokenize=True 来获取准确的 token 数
    # add_generation_prompt=False 因为我们要计算整个对话（包括最后的回答）的长度
    try:
        input_ids = tokenizer.apply_chat_template(
            messages, 
            tokenize=True, 
            add_generation_prompt=False
        )
        return len(input_ids) <= MAX_LEN
    except Exception as e:
        # 如果某条数据格式错误导致 template 应用失败，建议直接过滤掉以免训练报错
        print(f"Skipping sample due to error: {e}")
        return False

In [ ]:
dataset_cot_train = dataset_cot_train.filter(
    is_within_length_limit, 
    num_proc=1, 
)
dataset_cot_test = dataset_cot_test.filter(
    is_within_length_limit, 
    num_proc=1, 
)

In [ ]:
dataset_cot_train = dataset_cot_train.map(
    format_deepseek_r1,
    num_proc=1,  # 根据 CPU 核心数调整
)
dataset_cot_test = dataset_cot_test.map(
    format_deepseek_r1,
    num_proc=1,  # 根据 CPU 核心数调整
)

print(dataset_cot_train[0]['messages'])

# Part 3 Supervised Fine-Tuning

In [ ]:
# 3.1 配置LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,  # LoRA秩
    lora_alpha=32,  # 缩放参数
    target_modules=[  # Qwen 模块名
        "q_proj",     # query投影
        "k_proj",     # key投影
        "v_proj",     # value投影
        "o_proj",     # output投影
        "gate_proj",  # gate投影（MLP层）
        "up_proj",    # up投影（MLP层）
        "down_proj",  # down投影（MLP层）
    ],
    lora_dropout=0.1,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 3.2 配置训练参数
# 有效批大小 = per_device_batch_size * gradient_accumulation_steps * num_gpus
total_steps = (len(dataset_cot_train) * NUM_TRAIN_EPOCHS) // (BATCH_SIZE * ACCU_STEPS * NUM_GPUS)
warmup_steps = int(0.1 * total_steps)

training_args = TrainingArguments(
    output_dir=LORA_COT_DIR,    
    
    # 核心训练配置
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCU_STEPS,
    
    # 优化器与调度
    learning_rate=2e-4,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    optim="adamw_torch",
    
    # 精度配置
    bf16=True,  # 确保您的 GPU 支持 bf16 (Ampere架构及以上)，否则改为 fp16=True
    
    # 日志与报告
    # 日志保存目录 默认为 output_dir/runs
    logging_steps=10,
    report_to="tensorboard", # 确保已安装 tensorboard
    logging_first_step=True,
    
    # 保存策略
    save_steps=200,
    save_total_limit=3,
    save_strategy="steps",   # 显式指定保存策略 (新版推荐)
    
    # 评估策略
    eval_strategy=IntervalStrategy.STEPS, # 替代旧的 IntervalStrategy.STEPS，直接用字符串更稳妥
    eval_steps=200,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True, # 通常建议开启，以便训练结束后自动加载最优模型
    
    # 数据加载
    remove_unused_columns=False, # 使用 formatting_func 时必须为 False
    dataloader_num_workers=4,    # 建议设置，加速数据加载 (根据CPU核心数调整)
    
    # 其他
    disable_tqdm=False,
    
)

In [ ]:
# 3.3 创建SFTTrainer
# TRL 0.29.0 版本中，
# - SFTTrainer 默认行为就是只对 Assistant（助手）的回复部分计算 Loss，而忽略 User（用户）的输入部分。
# - 必须使用 messages 列，并利用传入的 processing_class (tokenizer) 调用 apply_chat_template 进行格式化。
# - 不再需要指定列名：因为列名被固定为 messages
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,     # Trainer 会自动调用 apply_chat_template
    train_dataset=dataset_cot_train,
    eval_dataset=dataset_cot_test,
    
    args=training_args,
)

In [ ]:
# 3.4 开始训练
print("开始训练...")
trainer.train()

In [ ]:
# 3.5 保存模型
# 这会自动保存：
# - LoRA模型权重 (adapter_model.bin)
# - tokenizer
# - training_args.bin
# - 配置文件
trainer.save_model(LORA_COT_FINAL)  
